# Back-test analysis

Thin wrapper over `ratesresearch.backtest_analysis`: reconciles a Java run's attribution, renders the daily tables, and recomputes the statistics of the ledger record of P5-001 from its daily P&L, reported across the declared band.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import SVG, Markdown, display

from ratesresearch import backtest_analysis as ba
from ratesresearch import presets

ROOT = Path.cwd()
while not (ROOT / "data" / "presets.yaml").is_file():
    ROOT = ROOT.parent
PRESETS = ROOT / "data" / "presets.yaml"


def show(df: pd.DataFrame) -> None:
    display(Markdown(ba.to_markdown(df)))

In [ ]:
from ratesresearch import prereg

run = ba.load_run(ROOT / "data/fixtures/backtest/fitting/2026-01-05")
display(Markdown("Reconciliation failures: " + str(ba.reconcile(run))))
display(Markdown(ba.daily_tables(run)))

In [ ]:
record = next(e for e in prereg.read_ledger(ROOT / "data/ledger/ledger.jsonl") if e["prereg_id"] == "P5-001")
display(Markdown(f"P5-001 {record['scope']} look {record['look']}: **{record['verdict']}**"))
show(ba.experiment_table(record))
re = ba.recompute_experiment(record)
show(pd.DataFrame([{"test": k, **{n: round(v, 6) for n, v in r.items()}} for k, r in re.items()]))